# Notebook 4: Cost-Quality Pareto Frontier

**Goal:** Find the most cost-efficient RAG strategies.  
**Questions:**
- Which experiments sit on the Pareto frontier?
- What is the marginal cost of each quality improvement?
- Which cost component dominates: embedding, retrieval overhead, or generation?
- What would each strategy cost at 1000 queries/day?

**Output:** `pareto_frontier.json`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from analysis.utils import (
    load_results, results_to_summary_df, composite_score,
    pareto_frontier, apply_plot_style, save_output
)
apply_plot_style()

In [ ]:
results = load_results()
df = results_to_summary_df(results)
df['composite'] = composite_score(df)

# Cost per query (total cost / number of queries)
n_queries = [len(r['per_query_results']) for r in results]
df['n_queries'] = n_queries
df['cost_per_query'] = df['total_cost_usd'] / df['n_queries'].clip(lower=1)
df[['experiment_id', 'composite', 'total_cost_usd', 'cost_per_query']].round(4)

## 1. Pareto Frontier Plot

In [ ]:
pareto_df = pareto_frontier(df, quality_col='composite', cost_col='cost_per_query')

fig = px.scatter(
    df,
    x='cost_per_query', y='composite',
    text='experiment_id',
    title='Cost vs Quality — All 10 Experiments',
    labels={'cost_per_query': 'Cost per Query (USD)', 'composite': 'Composite Quality Score'},
    width=900, height=550
)
# Overlay Pareto frontier line
pareto_sorted = pareto_df.sort_values('cost_per_query')
fig.add_trace(go.Scatter(
    x=pareto_sorted['cost_per_query'], y=pareto_sorted['composite'],
    mode='lines', name='Pareto frontier',
    line=dict(color='red', dash='dash')
))
fig.update_traces(textposition='top center')
fig.show()

print('Pareto-optimal experiments:')
print(pareto_df[['experiment_id', 'composite', 'cost_per_query']].to_string(index=False))

## 2. Marginal Cost of Quality Improvement

In [ ]:
pareto_sorted = pareto_df.sort_values('composite').reset_index(drop=True)
pareto_sorted['quality_gain'] = pareto_sorted['composite'].diff()
pareto_sorted['cost_increase'] = pareto_sorted['cost_per_query'].diff()
pareto_sorted['marginal_cost_per_quality_point'] = (
    pareto_sorted['cost_increase'] / pareto_sorted['quality_gain'].clip(lower=1e-6)
)

print('Marginal cost to gain each quality increment:')
display_cols = ['experiment_id', 'composite', 'cost_per_query', 'marginal_cost_per_quality_point']
print(pareto_sorted[display_cols].round(5).to_string(index=False))

## 3. Production Cost Projection (1000 queries/day)

In [ ]:
DAILY_QUERIES = 1000
DAYS_PER_MONTH = 30

df['daily_cost_usd']   = df['cost_per_query'] * DAILY_QUERIES
df['monthly_cost_usd'] = df['daily_cost_usd'] * DAYS_PER_MONTH

projection = df[['experiment_id', 'composite', 'cost_per_query', 'daily_cost_usd', 'monthly_cost_usd']].sort_values('monthly_cost_usd')
projection = projection.round(4)
print(f'Production cost projection at {DAILY_QUERIES} queries/day:')
print(projection.to_string(index=False))

In [ ]:
pareto_records = pareto_df[['experiment_id', 'composite', 'cost_per_query']].to_dict('records')
save_output(pareto_records, 'pareto_frontier.json')

projection_records = df[['experiment_id', 'composite', 'cost_per_query', 'monthly_cost_usd']].to_dict('records')
save_output(projection_records, 'cost_projection.json')
print('Saved.')